In [1]:
!pip uninstall -y massi
!pip cache purge

Found existing installation: massi 0.1.0
Uninstalling massi-0.1.0:
  Successfully uninstalled massi-0.1.0
Files removed: 6 (1.2 MB)


In [2]:
!cd .. && pip install -v . --group=dev --group=observability --group=sandbox

Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
Processing /home/jovyan/work
  Running command installing build dependencies
  Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
    Obtaining dependency information for setuptools>=40.8.0 from https://files.pythonhosted.org/packages/9d/76/f789f7a86709c6b087c5a2f52f911838cad707cc613162401badc665acfe/setuptools-82.0.1-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.0 MB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.5 MB/s  0:00:00
  Installing build dependencies ... done
  Running command Getting requirements to build wheel
  running egg_info
  writing massi.egg-info/PKG-INFO
  writing dependency_links to massi.egg-info/dependency_links.txt
  writing requirements to massi.egg-info/requires.txt
  writing top-level names to massi.egg-info/top_level.txt
  reading manifest file 'massi.egg-info/SOURCES.txt'
  writing manifest file

In [3]:
# Setup observability
USE_OBSERVABILITY = True

if USE_OBSERVABILITY:
    import os
    from lmnr import Laminar
    
    Laminar.initialize(
        project_api_key=os.environ["LMNR_PROJECT_API_KEY"],
        base_url=f"http://{os.environ["HOST_MACHINE_URL_CORE"]}",
        http_port=int(os.environ["LMNR_PROJECT_HTTP_PORT"]),
        grpc_port=int(os.environ["LMNR_PROJECT_GRPC_PORT"]),
    )

1. **We can use the relay model for the sake of debugging convenience**

In [4]:
from massi.agents.massi import MassiScenarioFactory, Config, Dependencies, StandardIOPromptClient

In [5]:
from massi.runtimes.jupyter.ui import JupyterTextWidgetIOPromptClient

In [6]:
from massi.agents.massi import create_ollama_llm

base_config = Config(**{"model_url": "http://host.docker.internal:11434", "model_name": "lfm2.5-thinking"})
base_deps = Dependencies(llm_client=create_ollama_llm(base_config), prompt_client=JupyterTextWidgetIOPromptClient())

In [7]:
USE_RELAY_AI = False

if USE_RELAY_AI:
    from massi.agents.stunts import RelayChatModel
    relay_model = RelayChatModel()
    relay_deps = Dependencies(llm_client=RelayChatModel(), prompt_client=JupyterTextWidgetIOPromptClient())

In [8]:
USE_OPEN_AI = True

if USE_OPEN_AI:
    from massi.agents.massi import create_openai_llm
    openai_config = Config(**{"api_key": os.environ["OPENAI_API_TOKEN"], "model_name": "gpt-4.1-nano"})
    openai_deps = Dependencies(llm_client=create_openai_llm(openai_config), prompt_client=JupyterTextWidgetIOPromptClient())

In [9]:
if USE_RELAY_AI:
    print("... using mocking dependencies ...")
    factory = MassiScenarioFactory(base_config)
    graph, _ = factory.resource_grouping_graph(relay_deps)
elif USE_OPEN_AI:
    print("... using OpenAI dependencies ...")
    factory = MassiScenarioFactory(openai_config)
    graph, _ = factory.resource_grouping_graph(openai_deps)
else:
    print("... using standard dependencies ...")
    factory = MassiScenarioFactory(base_config)
    graph, _ = factory.resource_grouping_graph(base_deps)

... using OpenAI dependencies ...


In [10]:
import grandalf

print(graph.get_graph().draw_mermaid())
graph.get_graph().print_ascii()

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	scan_directory(scan_directory)
	score_relevancy(score_relevancy)
	group_resources(group_resources)
	resources_to_description_file(resources_to_description_file)
	run_metadata_inference(run_metadata_inference)
	collect_metadata(collect_metadata)
	__end__([<p>__end__</p>]):::last
	__start__ --> scan_directory;
	group_resources --> resources_to_description_file;
	resources_to_description_file -.-> run_metadata_inference;
	run_metadata_inference --> collect_metadata;
	scan_directory -.-> score_relevancy;
	score_relevancy --> group_resources;
	collect_metadata --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

          +-----------+            
          | __start__ |            
          +-----------+            
                *                  
                *                  
                *                  
     

In [11]:
import random

thread = {"configurable": {"thread_id": str(random.randint(1, 100))}}

In [12]:
import pprint

In [13]:
from massi.runtimes.flow import stream_until_iterruption

In [14]:
from unittest.mock import patch

MOCK_RUN = False

In [15]:
# stream_until_iterruption(graph, {}, thread)
stream_until_iterruption(graph, {}, config=thread, vebose=True)

{'messages': [], 'resource': GroupedGisResourceFile(file_location='data/layer_data/WATER_SYSTEMS_IN_KENTUCKY.csv', filename='WATER_SYSTEMS_IN_KENTUCKY.csv', extension='.csv', author='jovyan', gis_relevancy_score=0.8, resource_name='WATER_SYSTEMS_IN_KENTUCKY', group_score=0.9333333333333332)}
{'messages': [], 'resource': GroupedGisResourceFile(file_location='data/layer_data/hot-spot_map.csv', filename='hot-spot_map.csv', extension='.csv', author='jovyan', gis_relevancy_score=0.7, resource_name='hot-spot_map', group_score=0.8666666666666667)}
{'messages': [], 'resource': GroupedGisResourceFile(file_location='data/layer_data/PFAS sampling and purchasing and intake detail.csv', filename='PFAS sampling and purchasing and intake detail.csv', extension='.csv', author='jovyan', gis_relevancy_score=0.8, resource_name='PFAS sampling and purchasing and intake detail', group_score=0.75)}
{'messages': [], 'resource': GroupedGisResourceFile(file_location='data/layer_data/waterIntake.csv', filename='

RuntimeError: error checking inheritance of Metadata (type: TypeAliasType)

In [ ]:
import pandas as pd
import duckdb as db

In [ ]:
groups_df = pd.read_csv(graph.get_state(thread).values["raw_resource_dataframe_file"])

In [ ]:
db.query("""
SELECT * 
FROM groups_df
""").to_df()

In [ ]:
properties_df = pd.read_csv(graph.get_state(thread).values["processed_resource_dataframe_file"])

In [ ]:
db.query("""
SELECT * 
FROM properties_df
""").to_df()